# 数据处理与格式转换

本notebook用于处理interim文件夹中的数据，主要完成以下任务：

1. 日期格式转换：将YYYY-MM格式转换为YYYY-MM-DD（使用月末日期）
2. 删除指定列：移除所有以'supplyment'开头的列
3. 数据完整性验证：与原始数据进行比对确保数据完整性

In [1]:
# 导入必要的库
import pandas as pd
import numpy as np
import calendar
from pathlib import Path
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# 设置数据路径
RAW_DATA_PATH = '../data/raw'
INTERIM_DATA_PATH = '../data/interim'
PROCESSED_DATA_PATH = '../data/processed'

# 创建processed目录（如果不存在）
Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [2]:
# 读取interim文件夹中的数据
print("=== 读取interim文件夹中的数据 ===")
import_data = pd.read_csv(os.path.join(INTERIM_DATA_PATH, 'combined_import_data.csv'))
export_data = pd.read_csv(os.path.join(INTERIM_DATA_PATH, 'combined_export_data.csv'))

print(f"Import数据形状：{import_data.shape}")
print(f"Export数据形状：{export_data.shape}")

# 显示列名
print("\nImport数据列名：")
print(import_data.columns.tolist())
print("\nExport数据列名：")
print(export_data.columns.tolist())

=== 读取interim文件夹中的数据 ===
Import数据形状：(1783, 11)
Export数据形状：(1164, 11)

Import数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']

Export数据列名：
['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']


In [4]:
# 检查列名（包含完整列名和数据类型信息）
print("\nImport数据详细信息：")
print(import_data.info())
print("\nExport数据详细信息：")
print(export_data.info())


Import数据详细信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1783 entries, 0 to 1782
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Date of data            1776 non-null   float64
 1   Commodity code          1783 non-null   int64  
 2   Commodity               1783 non-null   object 
 3   Trading partner code    1783 non-null   int64  
 4   Trading partner         1783 non-null   object 
 5   Quantity                1776 non-null   object 
 6   Unit                    1776 non-null   object 
 7   Supplementary Quantity  1769 non-null   float64
 8   Supplementary Unit      1769 non-null   object 
 9   US dollar               1769 non-null   object 
 10  source_file             1769 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 153.4+ KB
None

Export数据详细信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 11 columns)

In [13]:
# 日期格式转换函数
def convert_to_last_day(date_value):
    """
    将整数格式的日期转换为YYYY-MM-DD格式，使用每月的最后一天
    例如：202001 -> 2020-01-31
    """
    try:
        # 检查是否为NaN或None
        if pd.isna(date_value):
            return None  # 返回None而不是字符串，这样在DataFrame中会显示为NaN
        
        # 如果是浮点数，转换为整数
        if isinstance(date_value, float):
            date_value = int(date_value)
        
        # 提取年份（前4位）和月份（后2位）
        year = date_value // 100
        month = date_value % 100
        
        # 获取该月的最后一天
        last_day = calendar.monthrange(year, month)[1]
        
        # 返回格式化的日期字符串
        return f"{year}-{month:02d}-{last_day:02d}"
    except Exception as e:
        print(f"处理日期 {date_value} 时出错: {str(e)}")
        return None  # 出错时返回None

# 检查是否已经处理过日期列
if 'date' in import_data.columns:
    print("注意：数据已经包含 'date' 列，重新加载数据...")
    import_data = pd.read_csv(os.path.join(INTERIM_DATA_PATH, 'combined_import_data.csv'))
    export_data = pd.read_csv(os.path.join(INTERIM_DATA_PATH, 'combined_export_data.csv'))
    print("数据已重新加载")

# 查找日期列名
print("\n当前列名：")
print(f"Import列: {import_data.columns.tolist()}")
print(f"Export列: {export_data.columns.tolist()}")

# 查找日期列
date_col_import = 'Date of data'
date_col_export = 'Date of data'

# 检查空值情况
print(f"\nImport数据的 '{date_col_import}' 列空值数量：{import_data[date_col_import].isna().sum()}")
print(f"Export数据的 '{date_col_export}' 列空值数量：{export_data[date_col_export].isna().sum()}")

# 显示前几行数据
print(f"\nImport数据的 '{date_col_import}' 列前5行：")
print(import_data[date_col_import].head())
print(f"\nExport数据的 '{date_col_export}' 列前5行：")
print(export_data[date_col_export].head())

# 处理Import数据的日期
print("\n开始处理Import数据的日期...")
import_data['date'] = import_data[date_col_import].apply(convert_to_last_day)
import_data = import_data.drop(date_col_import, axis=1)
# 将date列移到第一列
import_data = import_data[['date'] + [col for col in import_data.columns if col != 'date']]

# 处理Export数据的日期
print("开始处理Export数据的日期...")
export_data['date'] = export_data[date_col_export].apply(convert_to_last_day)
export_data = export_data.drop(date_col_export, axis=1)
# 将date列移到第一列
export_data = export_data[['date'] + [col for col in export_data.columns if col != 'date']]

print("\n=== 日期格式转换完成 ===")
print("\n处理后的列名：")
print(f"Import列: {import_data.columns.tolist()}")
print(f"Export列: {export_data.columns.tolist()}")

# 检查转换后的空值
print(f"\n转换后Import数据的 'date' 列空值数量：{import_data['date'].isna().sum()}")
print(f"转换后Export数据的 'date' 列空值数量：{export_data['date'].isna().sum()}")

print("\n前10行Import数据的日期：")
print(import_data['date'].head(10))
print("\n前10行Export数据的日期：")
print(export_data['date'].head(10))

# 检查日期范围（排除NaN值）
print("\nImport数据日期范围（排除空值）：")
valid_import_dates = import_data['date'].dropna()
if len(valid_import_dates) > 0:
    print(f"最早日期：{valid_import_dates.min()}")
    print(f"最晚日期：{valid_import_dates.max()}")
else:
    print("没有有效的日期数据")

print("\nExport数据日期范围（排除空值）：")
valid_export_dates = export_data['date'].dropna()
if len(valid_export_dates) > 0:
    print(f"最早日期：{valid_export_dates.min()}")
    print(f"最晚日期：{valid_export_dates.max()}")
else:
    print("没有有效的日期数据")

注意：数据已经包含 'date' 列，重新加载数据...
数据已重新加载

当前列名：
Import列: ['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']
Export列: ['Date of data', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dollar', 'source_file']

Import数据的 'Date of data' 列空值数量：7
Export数据的 'Date of data' 列空值数量：0

Import数据的 'Date of data' 列前5行：
0   202401.00
1   202401.00
2   202401.00
3   202401.00
4   202401.00
Name: Date of data, dtype: float64

Export数据的 'Date of data' 列前5行：
0    202101
1    202101
2    202101
3    202101
4    202101
Name: Date of data, dtype: int64

开始处理Import数据的日期...
开始处理Export数据的日期...

=== 日期格式转换完成 ===

处理后的列名：
Import列: ['date', 'Commodity code', 'Commodity', 'Trading partner code', 'Trading partner', 'Quantity', 'Unit', 'Supplementary Quantity', 'Supplementary Unit', 'US dolla

In [14]:
# 删除supplyment开头的列
import_cols_to_drop = [col for col in import_data.columns if col.lower().startswith('supplyment')]
export_cols_to_drop = [col for col in export_data.columns if col.lower().startswith('supplyment')]

# 删除列并显示信息
if import_cols_to_drop:
    import_data = import_data.drop(import_cols_to_drop, axis=1)
    print("\n从Import数据中删除的列：")
    print(import_cols_to_drop)

if export_cols_to_drop:
    export_data = export_data.drop(export_cols_to_drop, axis=1)
    print("\n从Export数据中删除的列：")
    print(export_cols_to_drop)

print("\n=== 列删除完成 ===")
print(f"Import数据新形状：{import_data.shape}")
print(f"Export数据新形状：{export_data.shape}")


=== 列删除完成 ===
Import数据新形状：(1783, 11)
Export数据新形状：(1164, 11)


In [15]:
# 读取原始数据进行比对
def load_csv_with_encoding(file_path):
    """
    尝试使用不同的编码读取CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            return df
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"无法使用已知编码读取文件：{file_path}")

# 读取原始数据
print("=== 开始读取原始数据进行比对 ===")
raw_import_files = glob.glob(os.path.join(RAW_DATA_PATH, '*Import.csv'))
raw_export_files = glob.glob(os.path.join(RAW_DATA_PATH, '*Export.csv'))

# 统计原始数据的记录总数
raw_import_total = 0
raw_export_total = 0

for f in raw_import_files:
    df = load_csv_with_encoding(f)
    raw_import_total += len(df)

for f in raw_export_files:
    df = load_csv_with_encoding(f)
    raw_export_total += len(df)

print("\n数据完整性检查:")
print(f"原始Import数据总记录数: {raw_import_total}")
print(f"处理后Import数据记录数: {len(import_data)}")
print(f"Import数据记录数差异: {raw_import_total - len(import_data)}")

print(f"\n原始Export数据总记录数: {raw_export_total}")
print(f"处理后Export数据记录数: {len(export_data)}")
print(f"Export数据记录数差异: {raw_export_total - len(export_data)}")

# 检查是否有空值
print("\n空值检查:")
print("\nImport数据空值统计:")
print(import_data.isnull().sum())
print("\nExport数据空值统计:")
print(export_data.isnull().sum())

=== 开始读取原始数据进行比对 ===

数据完整性检查:
原始Import数据总记录数: 1776
处理后Import数据记录数: 1783
Import数据记录数差异: -7

原始Export数据总记录数: 1164
处理后Export数据记录数: 1164
Export数据记录数差异: 0

空值检查:

Import数据空值统计:
date                       7
Commodity code             0
Commodity                  0
Trading partner code       0
Trading partner            0
Quantity                   7
Unit                       7
Supplementary Quantity    14
Supplementary Unit        14
US dollar                 14
source_file               14
dtype: int64

Export数据空值统计:
date                      0
Commodity code            0
Commodity                 0
Trading partner code      0
Trading partner           0
Quantity                  0
Unit                      0
Supplementary Quantity    0
Supplementary Unit        0
US dollar                 0
source_file               0
dtype: int64

数据完整性检查:
原始Import数据总记录数: 1776
处理后Import数据记录数: 1783
Import数据记录数差异: -7

原始Export数据总记录数: 1164
处理后Export数据记录数: 1164
Export数据记录数差异: 0

空值检查:

Import数据空值统计:
date   

In [16]:
# 保存处理后的数据
output_import_path = os.path.join(PROCESSED_DATA_PATH, 'processed_import_data.csv')
output_export_path = os.path.join(PROCESSED_DATA_PATH, 'processed_export_data.csv')

import_data.to_csv(output_import_path, index=False, encoding='utf-8')
export_data.to_csv(output_export_path, index=False, encoding='utf-8')

print("=== 数据处理完成 ===")
print(f"处理后的Import数据已保存至：{output_import_path}")
print(f"处理后的Export数据已保存至：{output_export_path}")

=== 数据处理完成 ===
处理后的Import数据已保存至：../data/processed/processed_import_data.csv
处理后的Export数据已保存至：../data/processed/processed_export_data.csv
